---

# Unit 3 — Notebook 1: Advanced RAG & Retrieval Enhancement Techniques

## Introduction

In Unit 2, we built a **Naïve RAG** pipeline: embed a query → cosine search in FAISS → inject top-K docs → generate.

That works. But it has serious failure modes in production:

| Naïve RAG Problem | Example |
|---|---|
| **Vocabulary mismatch** | Query says "cardiac arrest"; document says "heart attack" → low cosine score despite same meaning |
| **Short, keyword-heavy queries** | "BM25 formula" → dense embeddings underperform on exact-match lookups |
| **Rare proper nouns / codes** | Product SKU `XR-7700-B` → embedding model has never seen it, maps it to near-zero space |
| **No term-frequency awareness** | Dense retrieval doesn't know how many times a term appears in a document |

**Advanced RAG** fixes these by upgrading the *retrieval* step with better algorithms.

### What We Will Build

```mermaid
graph LR
    Q["User Query"] --> S["Sparse Retrieval (BM25)"]
    Q --> D["Dense Retrieval (SBERT/ColBERT)"]
    S --> H["Hybrid Fusion (RRF)"]
    D --> H
    H --> G["LLM Generation"]
```

---

## Part 1a: Why Naïve RAG Fails — A Live Demonstration

In [6]:
# Install all dependencies for this notebook
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-community langchain-huggingface numpy --quiet

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

# If .env is missing, manually enter your key
if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Enter your Google API Key: ··········
Setup complete.


### Our Test Corpus

We will use a small corpus of 5 documents about AI topics. This corpus is deliberately designed to expose naïve RAG's weaknesses.

In [8]:
# Our knowledge base — 5 documents
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",           # doc_0
    "BERT is a bidirectional encoder trained using masked language modelling.",               # doc_1
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",  # doc_2
    "Gradient descent is an optimization technique used to minimize the loss function.",      # doc_3
    "Neural networks learn by adjusting weights through backpropagation.",                    # doc_4
]

# Our query: deliberately uses a synonym that dense models handle OK, but BM25 struggles with
query_vocab_mismatch = "How do neural nets learn?"

# Our query: deliberately uses an exact keyword that BM25 excels at
query_exact_match = "BM25 term frequency"

print(f"Corpus size: {len(corpus)} documents")
print(f"Query 1 (semantic): '{query_vocab_mismatch}'")
print(f"Query 2 (keyword):  '{query_exact_match}'")

Corpus size: 5 documents
Query 1 (semantic): 'How do neural nets learn?'
Query 2 (keyword):  'BM25 term frequency'


In [9]:
# --- Naïve Dense-Only Retrieval ---
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_embeddings = model.encode(corpus, convert_to_numpy=True)

def dense_retrieve(query, top_k=3):
    q_vec = model.encode([query], convert_to_numpy=True)[0]
    scores = corpus_embeddings @ q_vec / (
        np.linalg.norm(corpus_embeddings, axis=1) * np.linalg.norm(q_vec)
    )
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i], corpus[i]) for i in ranked]

print("=== Dense Retrieval Results ===")
print(f"\nQuery: '{query_vocab_mismatch}'")
for rank, (idx, score, doc) in enumerate(dense_retrieve(query_vocab_mismatch), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

print(f"\nQuery: '{query_exact_match}'")
for rank, (idx, score, doc) in enumerate(dense_retrieve(query_exact_match), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== Dense Retrieval Results ===

Query: 'How do neural nets learn?'
  Rank 1 [doc_4] score=0.7550: Neural networks learn by adjusting weights through backpropagation.
  Rank 2 [doc_3] score=0.3656: Gradient descent is an optimization technique used to minimize the los
  Rank 3 [doc_1] score=0.2918: BERT is a bidirectional encoder trained using masked language modellin

Query: 'BM25 term frequency'
  Rank 1 [doc_2] score=0.5714: The BM25 algorithm ranks documents based on term frequency and inverse
  Rank 2 [doc_0] score=0.1448: Transformers use self-attention mechanisms to process sequences in par
  Rank 3 [doc_4] score=0.1129: Neural networks learn by adjusting weights through backpropagation.


### Observation

- For Query 1 (semantic), dense retrieval does well — it understands "neural nets learn" ≈ "backpropagation / weights".
- For Query 2 (keyword exact-match), dense retrieval may **not** rank `doc_2` (the BM25 document) at the top — the embedding model sees "BM25 term frequency" as a technical phrase it doesn't strongly embed near the correct document.

This is the core gap. Let's now see the other side of the coin: **Sparse Retrieval**.

---

## Part 1b: Sparse Retrieval — BM25

### What is BM25?

**BM25 (Best Match 25)** is the gold standard sparse retrieval algorithm. It is still used inside Elasticsearch and OpenSearch today.

It evolved from TF-IDF (Term Frequency × Inverse Document Frequency), but fixes two of its problems:

| Problem in TF-IDF | BM25 Fix |
|---|---|
| Score grows unboundedly with more term repetitions | **Saturation**: Term frequency is dampened |
| Long documents unfairly score higher (more words = more matches) | **Length Normalization**: Penalizes documents longer than average |

### BM25 Formula

For query $Q$ with terms $q_1, q_2, \ldots, q_n$ and document $D$:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Where:
- $f(q_i, D)$ = term frequency of $q_i$ in document $D$
- $|D|$ = length of document $D$ (in words)
- $\text{avgdl}$ = average document length across the corpus
- $k_1 \in [1.2, 2.0]$ = controls term frequency saturation (default: **1.5**)
- $b \in [0, 1]$ = controls length normalization strength (default: **0.75**)
- $\text{IDF}(q_i) = \log\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$, where $N$ = total docs, $n(q_i)$ = docs containing $q_i$

In [10]:
from rank_bm25 import BM25Okapi

# BM25 works on tokenized (word-split) documents
tokenized_corpus = [doc.lower().split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i], corpus[i]) for i in ranked]

print("=== BM25 Retrieval Results ===")
print(f"\nQuery: '{query_vocab_mismatch}'")
for rank, (idx, score, doc) in enumerate(bm25_retrieve(query_vocab_mismatch), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

print(f"\nQuery: '{query_exact_match}'")
for rank, (idx, score, doc) in enumerate(bm25_retrieve(query_exact_match), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

=== BM25 Retrieval Results ===

Query: 'How do neural nets learn?'
  Rank 1 [doc_4] score=1.2259: Neural networks learn by adjusting weights through backpropagation.
  Rank 2 [doc_3] score=0.0000: Gradient descent is an optimization technique used to minimize the los
  Rank 3 [doc_2] score=0.0000: The BM25 algorithm ranks documents based on term frequency and inverse

Query: 'BM25 term frequency'
  Rank 1 [doc_2] score=2.9625: The BM25 algorithm ranks documents based on term frequency and inverse
  Rank 2 [doc_4] score=0.0000: Neural networks learn by adjusting weights through backpropagation.
  Rank 3 [doc_3] score=0.0000: Gradient descent is an optimization technique used to minimize the los


### Observation

- BM25 excels at Query 2 (exact keyword match) — it immediately finds `doc_2` with the highest score because the terms "BM25" and "term" and "frequency" all appear in it.
- BM25 **fails** at Query 1 (semantic) — it scores 0 for documents that don't share exact words with the query ("neural nets learn" has no token overlap with "backpropagation" or "weights").

**Neither BM25 nor Dense alone is sufficient. We need both.**

---

## Part 1c: Dense Retrieval — SBERT and ColBERT

### SBERT (Sentence-BERT)

SBERT fine-tunes BERT with a **siamese network** objective so that semantically similar sentences produce similar embeddings.

```mermaid
graph LR
    A["Sentence A"] --> BERT_A["BERT Encoder"]
    B["Sentence B"] --> BERT_B["BERT Encoder (shared weights)"]
    BERT_A --> Pool_A["Mean Pooling → Vector A"]
    BERT_B --> Pool_B["Mean Pooling → Vector B"]
    Pool_A --> Sim["Cosine Similarity"]
    Pool_B --> Sim
```

- **Bi-encoder**: Document and query are encoded **independently** → fast at retrieval time (pre-compute doc embeddings once).
- Used in our previous FAISS-based RAG (all-MiniLM-L6-v2 is an SBERT model).

### ColBERT (Contextualized Late Interaction over BERT)

ColBERT is a more powerful dense retrieval approach. Instead of compressing a sentence into **one vector**, it produces **one vector per token**.

```mermaid
graph TD
    subgraph "SBERT (Bi-Encoder)"
        QS["Query tokens"] --> EQ["Encoder"] --> VQ["1 vector [CLS]"]
        DS["Doc tokens"] --> ED["Encoder"] --> VD["1 vector [CLS]"]
        VQ -- "dot product" --> ScoreS["Score"]
        VD -- "dot product" --> ScoreS
    end
    subgraph "ColBERT (Late Interaction)"
        QC["Query tokens: q1 q2 q3"] --> EQC["Encoder"] --> VQC["3 token vectors"]
        DC["Doc tokens: d1 d2 d3 d4"] --> EDC["Encoder"] --> VDC["4 token vectors"]
        VQC -- "MaxSim: each qi scores vs all dj" --> ScoreC["Sum of MaxSims"]
        VDC --> ScoreC
    end
```

#### ColBERT MaxSim Scoring

$$\text{ColBERT}(Q, D) = \sum_{i \in Q} \max_{j \in D} \left( \vec{q}_i \cdot \vec{d}_j \right)$$

For each query token $q_i$, find the **maximum similarity** across all document tokens $d_j$, then sum.

**Why is this better?**
- Each query word can independently "find" the most relevant document word.
- The word "attack" in the query can match "assault" in the document via contextual embeddings.
- Much richer than a single dot product.

In [11]:
# ColBERT-style Late Interaction — implemented from scratch to understand the mechanism
# We use all-MiniLM as our encoder (true ColBERT uses a dedicated model, but the math is identical)

from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def colbert_score(query: str, document: str) -> float:
    """
    ColBERT MaxSim scoring.
    Each query token finds its best matching document token.
    Score = sum of those max similarities.
    """
    query_tokens = query.lower().split()       # tokenize (simplified: word-level)
    doc_tokens   = document.lower().split()

    # Encode each token as a vector
    q_vecs = encoder.encode(query_tokens, convert_to_numpy=True)   # shape: (|Q|, dim)
    d_vecs = encoder.encode(doc_tokens,   convert_to_numpy=True)   # shape: (|D|, dim)

    # Normalize
    q_vecs = q_vecs / np.linalg.norm(q_vecs, axis=1, keepdims=True)
    d_vecs = d_vecs / np.linalg.norm(d_vecs, axis=1, keepdims=True)

    # Similarity matrix: (|Q|, |D|)
    sim_matrix = q_vecs @ d_vecs.T

    # MaxSim: for each query token, take max across all doc tokens
    max_sims = sim_matrix.max(axis=1)    # shape: (|Q|,)

    return float(max_sims.sum())

print("=== ColBERT-style Late Interaction Scores ===")
query = "How do neural nets learn?"
print(f"Query: '{query}'\n")
scores = []
for i, doc in enumerate(corpus):
    s = colbert_score(query, doc)
    scores.append(s)
    print(f"  doc_{i} score={s:.4f}: {doc[:70]}")

best = np.argmax(scores)
print(f"\nTop document: doc_{best} — '{corpus[best]}'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== ColBERT-style Late Interaction Scores ===
Query: 'How do neural nets learn?'

  doc_0 score=1.8954: Transformers use self-attention mechanisms to process sequences in par
  doc_1 score=2.1215: BERT is a bidirectional encoder trained using masked language modellin
  doc_2 score=1.8315: The BM25 algorithm ranks documents based on term frequency and inverse
  doc_3 score=1.9923: Gradient descent is an optimization technique used to minimize the los
  doc_4 score=3.3560: Neural networks learn by adjusting weights through backpropagation.

Top document: doc_4 — 'Neural networks learn by adjusting weights through backpropagation.'


### Key Difference Summary

| Model | Representation | Speed | Quality |
|---|---|---|---|
| **BM25** | Sparse (bag-of-words) | Very fast | Great for exact keywords |
| **SBERT** | 1 dense vector per sentence | Fast (pre-encoded) | Great for semantic meaning |
| **ColBERT** | N dense vectors (1 per token) | Slower, more memory | Best of both — token-level semantics |

---

## Part 1d: Hybrid Retrieval — BM25 + SBERT with Reciprocal Rank Fusion

### The Problem with Simply Adding Scores

BM25 scores are in range [0, ∞] (e.g., `4.7`).  
SBERT cosine scores are in range [-1, 1] (e.g., `0.82`).  

You cannot add them directly — the scales are incompatible.

### Solution: Reciprocal Rank Fusion (RRF)

Instead of combining raw scores, RRF combines **rank positions**. This makes it scale-agnostic.

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + r(d)}$$

Where:
- $R$ = set of ranked lists (e.g., BM25 list, SBERT list)
- $r(d)$ = rank of document $d$ in list $r$ (1-indexed)
- $k$ = smoothing constant (default **60**) — prevents rank 1 from being overwhelmingly dominant

### Worked Numerical Example

Let's trace through the full calculation step by step with our 5-document corpus.

In [12]:
# ============================================================
# NUMERICAL EXAMPLE: Full BM25 + SBERT + RRF walkthrough
# Query: "How do neural nets learn?"
# ============================================================

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is a bidirectional encoder trained using masked language modelling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization technique used to minimize the loss function.",
    "Neural networks learn by adjusting weights through backpropagation.",
]

query = "How do neural nets learn?"

# ---- STEP 1: BM25 Scores ----
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(query.lower().split())

print("STEP 1: BM25 Raw Scores")
print("-" * 60)
for i, s in enumerate(bm25_scores):
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:60]}")

bm25_ranked = np.argsort(bm25_scores)[::-1]
bm25_ranks  = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}

print("\nBM25 Ranking (best → worst):", list(bm25_ranked))

STEP 1: BM25 Raw Scores
------------------------------------------------------------
  doc_0: 0.0000  |  Transformers use self-attention mechanisms to process sequen
  doc_1: 0.0000  |  BERT is a bidirectional encoder trained using masked languag
  doc_2: 0.0000  |  The BM25 algorithm ranks documents based on term frequency a
  doc_3: 0.0000  |  Gradient descent is an optimization technique used to minimi
  doc_4: 1.2259  |  Neural networks learn by adjusting weights through backpropa

BM25 Ranking (best → worst): [np.int64(4), np.int64(3), np.int64(2), np.int64(1), np.int64(0)]


In [13]:
# ---- STEP 2: SBERT Dense Scores ----
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

doc_vecs   = sbert.encode(corpus, convert_to_numpy=True)
query_vec  = sbert.encode([query], convert_to_numpy=True)[0]

doc_vecs_norm   = doc_vecs   / np.linalg.norm(doc_vecs,   axis=1, keepdims=True)
query_vec_norm  = query_vec  / np.linalg.norm(query_vec)

sbert_scores = doc_vecs_norm @ query_vec_norm

print("STEP 2: SBERT Cosine Scores")
print("-" * 60)
for i, s in enumerate(sbert_scores):
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:60]}")

sbert_ranked = np.argsort(sbert_scores)[::-1]
sbert_ranks  = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}

print("\nSBERT Ranking (best → worst):", list(sbert_ranked))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 2: SBERT Cosine Scores
------------------------------------------------------------
  doc_0: 0.2637  |  Transformers use self-attention mechanisms to process sequen
  doc_1: 0.2918  |  BERT is a bidirectional encoder trained using masked languag
  doc_2: 0.0809  |  The BM25 algorithm ranks documents based on term frequency a
  doc_3: 0.3656  |  Gradient descent is an optimization technique used to minimi
  doc_4: 0.7550  |  Neural networks learn by adjusting weights through backpropa

SBERT Ranking (best → worst): [np.int64(4), np.int64(3), np.int64(1), np.int64(0), np.int64(2)]


In [14]:
# ---- STEP 3: Reciprocal Rank Fusion ----

K = 60  # RRF smoothing constant

print("STEP 3: Reciprocal Rank Fusion (k=60)")
print("-" * 80)
print(f"{'Doc':<8} {'BM25 rank':<12} {'SBERT rank':<13} {'RRF(BM25)':<14} {'RRF(SBERT)':<14} {'Total RRF'}")
print("-" * 80)

rrf_scores = {}
for doc_id in range(len(corpus)):
    rb = bm25_ranks[doc_id]
    rs = sbert_ranks[doc_id]
    rrf_bm25  = 1.0 / (K + rb)
    rrf_sbert = 1.0 / (K + rs)
    total     = rrf_bm25 + rrf_sbert
    rrf_scores[doc_id] = total
    print(f"  doc_{doc_id}  {rb:<12} {rs:<13} {rrf_bm25:<14.6f} {rrf_sbert:<14.6f} {total:.6f}")

# Final ranking
final_ranked = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

print("\n" + "=" * 80)
print("FINAL HYBRID RANKING")
print("=" * 80)
for rank, doc_id in enumerate(final_ranked, 1):
    print(f"  #{rank}  doc_{doc_id} (RRF={rrf_scores[doc_id]:.6f}): {corpus[doc_id]}")

STEP 3: Reciprocal Rank Fusion (k=60)
--------------------------------------------------------------------------------
Doc      BM25 rank    SBERT rank    RRF(BM25)      RRF(SBERT)     Total RRF
--------------------------------------------------------------------------------
  doc_0  5            4             0.015385       0.015625       0.031010
  doc_1  4            3             0.015625       0.015873       0.031498
  doc_2  3            5             0.015873       0.015385       0.031258
  doc_3  2            2             0.016129       0.016129       0.032258
  doc_4  1            1             0.016393       0.016393       0.032787

FINAL HYBRID RANKING
  #1  doc_4 (RRF=0.032787): Neural networks learn by adjusting weights through backpropagation.
  #2  doc_3 (RRF=0.032258): Gradient descent is an optimization technique used to minimize the loss function.
  #3  doc_1 (RRF=0.031498): BERT is a bidirectional encoder trained using masked language modelling.
  #4  doc_2 (RRF=0.0

### Interpreting the Numbers

Let's trace `doc_4` ("Neural networks learn by adjusting weights through backpropagation."):

- **BM25 rank**: Likely 1 or 2 (tokens "neural", "learn" match the query exactly)
- **SBERT rank**: Also likely 1 or 2 (semantically, this is the closest sentence)

RRF reward = `1/(60+1) + 1/(60+1)` = `0.01639 + 0.01639` = **`0.03279`**

A document that ranks **#1 in both lists** gets the maximum possible RRF score. A document that ranks in the middle of one but bottom of the other still gets partial credit — it won't be buried.

**The beauty of RRF**: It doesn't matter if BM25 gives scores of `4.7` and SBERT gives `0.8`. We only care about *positions in the ranked list*.

In [15]:
# ---- Comparison Table: BM25 vs SBERT vs Hybrid ----

print("COMPARISON: BM25 vs SBERT vs Hybrid (for query: 'How do neural nets learn?')")
print("=" * 90)
print(f"{'Rank':<6} {'BM25 top doc':<50} {'SBERT top doc':<50}")
print("-" * 90)
for i in range(len(corpus)):
    bm25_doc = corpus[bm25_ranked[i]][:45]
    sbert_doc = corpus[sbert_ranked[i]][:45]
    print(f"  #{i+1}   {bm25_doc:<50} {sbert_doc:<50}")

print()
print("Hybrid Final Ranking:")
for rank, doc_id in enumerate(final_ranked, 1):
    print(f"  #{rank}  {corpus[doc_id]}")

COMPARISON: BM25 vs SBERT vs Hybrid (for query: 'How do neural nets learn?')
Rank   BM25 top doc                                       SBERT top doc                                     
------------------------------------------------------------------------------------------
  #1   Neural networks learn by adjusting weights th      Neural networks learn by adjusting weights th     
  #2   Gradient descent is an optimization technique      Gradient descent is an optimization technique     
  #3   The BM25 algorithm ranks documents based on t      BERT is a bidirectional encoder trained using     
  #4   BERT is a bidirectional encoder trained using      Transformers use self-attention mechanisms to     
  #5   Transformers use self-attention mechanisms to      The BM25 algorithm ranks documents based on t     

Hybrid Final Ranking:
  #1  Neural networks learn by adjusting weights through backpropagation.
  #2  Gradient descent is an optimization technique used to minimize the loss fun

### Second Numerical Example — Keyword-Heavy Query

In [16]:
# Test with a keyword-heavy query where BM25 should dominate

query2 = "BM25 term frequency ranking"

bm25_scores2  = bm25.get_scores(query2.lower().split())
bm25_ranked2  = np.argsort(bm25_scores2)[::-1]
bm25_ranks2   = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked2)}

query2_vec    = sbert.encode([query2], convert_to_numpy=True)[0]
query2_norm   = query2_vec / np.linalg.norm(query2_vec)
sbert_scores2 = doc_vecs_norm @ query2_norm
sbert_ranked2 = np.argsort(sbert_scores2)[::-1]
sbert_ranks2  = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked2)}

rrf_scores2 = {}
for doc_id in range(len(corpus)):
    rrf_scores2[doc_id] = 1/(K + bm25_ranks2[doc_id]) + 1/(K + sbert_ranks2[doc_id])
final_ranked2 = sorted(rrf_scores2, key=rrf_scores2.get, reverse=True)

print(f"Query: '{query2}'\n")
print(f"{'Doc':<8} {'BM25 rank':<12} {'SBERT rank':<13} {'RRF Score':<12} Document")
print("-" * 90)
for doc_id in range(len(corpus)):
    print(f"  doc_{doc_id}  {bm25_ranks2[doc_id]:<12} {sbert_ranks2[doc_id]:<13} {rrf_scores2[doc_id]:.6f}   {corpus[doc_id][:60]}")

print("\nHybrid Final Ranking:")
for rank, doc_id in enumerate(final_ranked2, 1):
    print(f"  #{rank}  doc_{doc_id}: {corpus[doc_id]}")

Query: 'BM25 term frequency ranking'

Doc      BM25 rank    SBERT rank    RRF Score    Document
------------------------------------------------------------------------------------------
  doc_0  5            5             0.030769   Transformers use self-attention mechanisms to process sequen
  doc_1  4            3             0.031498   BERT is a bidirectional encoder trained using masked languag
  doc_2  1            1             0.032787   The BM25 algorithm ranks documents based on term frequency a
  doc_3  3            4             0.031498   Gradient descent is an optimization technique used to minimi
  doc_4  2            2             0.032258   Neural networks learn by adjusting weights through backpropa

Hybrid Final Ranking:
  #1  doc_2: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.
  #2  doc_4: Neural networks learn by adjusting weights through backpropagation.
  #3  doc_1: BERT is a bidirectional encoder trained using masked

### Why `k=60` Specifically?

The constant `k=60` was empirically determined in the original RRF paper (Cormack et al., 2009) to work well across many retrieval benchmarks. It prevents the #1 rank from dominating too heavily.

| k value | Effect |
|---|---|
| **k=0** | Rank 1 gets ∞ weight — effectively winner-takes-all |
| **k=1** | Rank 1: 0.5, Rank 2: 0.33 — very steep drop-off |
| **k=60** | Rank 1: 0.0164, Rank 2: 0.0161 — gentle, democratic |
| **k=1000** | All ranks are near-equal — no differentiation |

---

## Part 1e: Building a Full Hybrid Retriever (Reusable Function)

In [17]:
class HybridRetriever:
    """
    A production-style Hybrid Retriever combining BM25 (sparse) + SBERT (dense)
    using Reciprocal Rank Fusion.
    """

    def __init__(self, corpus: list[str], sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2", k: int = 60):
        self.corpus = corpus
        self.k = k

        # --- Build BM25 index ---
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25  = BM25Okapi(tokenized)

        # --- Build SBERT index ---
        self.sbert     = SentenceTransformer(sbert_model)
        doc_vecs       = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs  = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:
        # BM25
        bm25_scores  = self.bm25.get_scores(query.lower().split())
        bm25_ranked  = np.argsort(bm25_scores)[::-1]
        bm25_ranks   = {int(doc_id): rank+1 for rank, doc_id in enumerate(bm25_ranked)}

        # SBERT
        q_vec        = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec        = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(doc_id): rank+1 for rank, doc_id in enumerate(sbert_ranked)}

        # RRF Fusion
        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = 1/(self.k + bm25_ranks[doc_id]) + 1/(self.k + sbert_ranks[doc_id])

        final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": doc_id, "rrf_score": rrf[doc_id], "text": self.corpus[doc_id]}
            for doc_id in final
        ]


# Demo
retriever = HybridRetriever(corpus)

for q in ["How do neural nets learn?", "BM25 term frequency ranking", "attention in transformers"]:
    print(f"Query: '{q}'")
    results = retriever.retrieve(q, top_k=2)
    for r in results:
        print(f"  → doc_{r['doc_id']} (RRF={r['rrf_score']:.5f}): {r['text']}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: 'How do neural nets learn?'
  → doc_4 (RRF=0.03279): Neural networks learn by adjusting weights through backpropagation.
  → doc_3 (RRF=0.03226): Gradient descent is an optimization technique used to minimize the loss function.

Query: 'BM25 term frequency ranking'
  → doc_2 (RRF=0.03279): The BM25 algorithm ranks documents based on term frequency and inverse document frequency.
  → doc_4 (RRF=0.03226): Neural networks learn by adjusting weights through backpropagation.

Query: 'attention in transformers'
  → doc_0 (RRF=0.03279): Transformers use self-attention mechanisms to process sequences in parallel.
  → doc_4 (RRF=0.03226): Neural networks learn by adjusting weights through backpropagation.



---

## Summary

| Technique | Strength | Weakness | Library |
|---|---|---|---|
| **BM25** | Exact keywords, proper nouns, rare terms | No semantic understanding | `rank_bm25` |
| **SBERT** | Semantic similarity, paraphrases | Misses exact keyword matches | `sentence-transformers` |
| **ColBERT** | Token-level semantics, best quality | Slower, higher memory | `colbert-ai/colbert` |
| **Hybrid (RRF)** | Best of both worlds, scale-agnostic fusion | More compute than either alone | Custom (built above) |

### Key Takeaways
1. **Sparse retrieval** (BM25) = bag-of-words, term statistics, exact match power.
2. **Dense retrieval** (SBERT/ColBERT) = neural embeddings, semantic understanding.
3. **Hybrid retrieval** (BM25 + SBERT + RRF) = combines both by fusing rank lists, not raw scores.
4. **RRF formula**: `score(d) = Σ 1/(k + rank(d))` — simple, powerful, no hyperparameter tuning needed.

**Next**: Notebook 2 will improve the retrieved results further using **Re-Ranking** and **Query Expansion** before they reach the LLM.